# ProjectEcho — PPO Training

**Setup:**
1. Attach the `projectecho-training` dataset (your uploaded `training/` folder) as input data.
2. Set `VERSION` in Cell 2 (e.g. `"v2"`). Output goes to `/kaggle/working/{VERSION}/`.
3. To fine-tune from a previous model: attach that model's `best_model.zip` as a second dataset, then set `RESUME_MODEL` in Cell 4.

Trained checkpoints and TensorBoard logs are written to `/kaggle/working/{VERSION}/`.
Download `best_model.zip` from the Output tab when done. Rename it `v{N}_best_model.zip` locally to avoid confusion.

In [ ]:
# Install training dependencies from the repo requirements file
import subprocess, sys
TRAINING_DIR = "/kaggle/input/datasets"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "-r", f"{TRAINING_DIR}/requirements.txt",
], check=True)

In [ ]:
import sys, os

TRAINING_DIR = "/kaggle/input/datasets"
if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)

# Bump VERSION each training run to keep outputs separate
VERSION    = "v2"
OUTPUT_DIR = f"/kaggle/working/{VERSION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"sys.path OK. Version: {VERSION}  Output dir: {OUTPUT_DIR}")

In [ ]:
# Smoke-test: env imports and resets correctly
from env_wrappers import LeagueEnv
env = LeagueEnv(num_tribes=4, max_ticks=50)
obs, _ = env.reset(seed=42)
mask = env.action_masks()

assert obs.shape == env.observation_space.shape, \
    f"obs shape mismatch: expected {env.observation_space.shape}, got {obs.shape}"
assert mask.shape == (env.action_space.n,), \
    f"mask shape mismatch: expected ({env.action_space.n},), got {mask.shape}"
assert mask[11], "ACTION_REST (index 11) must always be valid after reset"

print(f"Smoke test passed — obs: {obs.shape}, valid actions: {mask.sum()}/{env.action_space.n}")

In [ ]:
# Run training.
#
# Kaggle free tier: 2 CPU cores, 13 GB RAM, 30 GPU hrs/week.
# --envs 4 is safe; drop to 2 if you hit OOM.
# --timesteps 500000 takes ~2-3 hours on P100 GPU.
#
# To fine-tune from a previous model:
#   1. Upload the previous best_model.zip as a Kaggle dataset (e.g. "projectecho-model-v1")
#   2. Attach it to this notebook as a second input dataset
#   3. Set RESUME_MODEL to its path (without .zip extension)
#
RESUME_MODEL = None  # e.g. "/kaggle/input/projectecho-model-v1/best_model"

cmd = [
    sys.executable,
    os.path.join(TRAINING_DIR, "train.py"),
    "--timesteps", "500000",
    "--envs",      "4",
    "--seed",      "42",
    "--output-dir", OUTPUT_DIR,
    "--win-rate",  "0.60",
    "--dummy-vec",
]
if RESUME_MODEL:
    cmd += ["--resume", RESUME_MODEL]
    print(f"Resuming from {RESUME_MODEL}.zip")
else:
    print("Starting fresh training run")

result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f"train.py exited with code {result.returncode} — check output above")

In [ ]:
# List saved checkpoints
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(path) // 1024
    print(f"{f:40s}  {size} KB")

In [ ]:
# Gauntlet: evaluate best_model over 100 episodes vs HeuristicPolicy opponents
# Runs AFTER training. Uses deterministic (greedy) policy — no exploration noise.
import numpy as np
from sb3_contrib import MaskablePPO
from env_wrappers import LeagueEnv

GAUNTLET_EPISODES = 100
model_path = os.path.join(OUTPUT_DIR, "best_model")

print(f"Loading {model_path}.zip ...")
gauntlet_model = MaskablePPO.load(model_path)

eval_env = LeagueEnv(num_tribes=4, max_ticks=200, domain_randomize=False, heuristic_opponents=True)

wins = 0
results = []

for ep in range(GAUNTLET_EPISODES):
    obs, _ = eval_env.reset(seed=ep)
    done = False
    while not done:
        masks = eval_env.action_masks()
        action, _ = gauntlet_model.predict(obs, action_masks=masks, deterministic=True)
        obs, _, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated

    info = eval_env.get_final_info()
    learner_tiles = info.get("learner_tiles", 0)
    opp_tiles     = info.get("opponent_tiles", [])
    alive         = info.get("learner_alive", False)
    win           = alive and learner_tiles > max(opp_tiles, default=0)
    wins         += int(win)
    results.append(dict(
        win=win, alive=alive,
        learner_tiles=learner_tiles,
        best_opp=max(opp_tiles, default=0),
    ))

win_rate     = wins / GAUNTLET_EPISODES
alive_rate   = sum(r["alive"] for r in results) / GAUNTLET_EPISODES
avg_learner  = np.mean([r["learner_tiles"] for r in results])
avg_best_opp = np.mean([r["best_opp"] for r in results])

print(f"\n{'='*45}")
print(f"Gauntlet  {VERSION}  ({GAUNTLET_EPISODES} eps vs HeuristicPolicy)")
print(f"{'='*45}")
print(f"  Win rate      : {win_rate:.1%}  ({wins}/{GAUNTLET_EPISODES})")
print(f"  Survival rate : {alive_rate:.1%}")
print(f"  Avg learner tiles  : {avg_learner:.1f}")
print(f"  Avg best-opp tiles : {avg_best_opp:.1f}")
print(f"{'='*45}")
print(f"\nDownload: {OUTPUT_DIR}/best_model.zip")
print(f"Rename to: v2_best_model.zip to keep versions separate")